# Notebook 04 — Benchmark Analysis

Load metrics from all three variants and produce five main visualisations:
1. Metric overview vs α (line plots)
2. Connectivity tax ratio curves
3. Heat maps (α vs Trotter steps)
4. Phase diagram (feasibility regions)
5. Summary table with threshold analysis

No simulations — this notebook only loads and visualises.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

from src.metrics import load_metrics
from src.analysis import metrics_to_dataframe, compute_tax_ratios

## Load all results

In [ ]:
ideal = load_metrics("../results/ideal_metrics.json")
unitary = load_metrics("../results/unitary_metrics.json")
dynamic = load_metrics("../results/dynamic_metrics.json")

all_metrics = ideal + unitary + dynamic
df = metrics_to_dataframe(all_metrics)
df = compute_tax_ratios(df)

print(f"Total records: {len(df)}")
print(f"Variants: {list(df['variant'].unique())}")
print(f"Alphas: {sorted(df['alpha'].unique())}")
print(f"Steps range: {df['n_steps'].min()} – {df['n_steps'].max()}")

---
## Plot 1 — Circuit Metrics vs α (1 Trotter step)

Four key metrics for all three variants.  Shows how the cost
landscape changes with interaction range.

In [ ]:
THETA_FIXED = np.pi / 4
sub1 = df[(df["n_steps"] == 1) & (np.isclose(df["theta"], THETA_FIXED))]

metrics_to_plot = ["two_q_depth", "two_q_count", "swap_count", "total_depth"]
titles = ["Two-Qubit Depth", "Two-Qubit Gate Count", "SWAP Count", "Total Depth"]

variant_styles = {
    "ideal":   {"color": "#2E7D32", "marker": "o", "ls": "-",  "label": "Ideal (all-to-all)"},
    "unitary": {"color": "#C62828", "marker": "s", "ls": "--", "label": "Heavy-hex unitary"},
    "dynamic": {"color": "#1565C0", "marker": "^", "ls": "-.", "label": "Heavy-hex dynamic"},
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for ax, metric, title in zip(axes.ravel(), metrics_to_plot, titles):
    for variant, style in variant_styles.items():
        data = sub1[sub1["variant"] == variant].sort_values("alpha")
        ax.plot(data["alpha"], data[metric], markersize=7, **style)
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

for ax in axes[1]:
    ax.set_xlabel(r"$\alpha$")

fig.suptitle("Circuit metrics vs power-law exponent  (N=6, 1 Trotter step)", fontsize=14)
fig.tight_layout()
fig.savefig("../results/plot1_metrics_vs_alpha.png", dpi=150)
plt.show()

---
## Plot 2 — Connectivity Tax Ratios

Tax = (variant 2Q depth) / (ideal 2Q depth).
A ratio of 1.0 means no overhead; higher = worse.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for variant, color, marker in [("unitary", "#C62828", "s"), ("dynamic", "#1565C0", "^")]:
    data = sub1[sub1["variant"] == variant].sort_values("alpha")
    tax = data["two_q_depth_tax"]
    ax.plot(data["alpha"], tax, f"{marker}-", color=color, label=variant, markersize=8)

ax.axhline(1.0, color="#2E7D32", ls="--", alpha=0.7, lw=2, label="ideal (ratio = 1)")
ax.set_xlabel(r"$\alpha$", fontsize=12)
ax.set_ylabel("2Q Depth Tax Ratio  (variant / ideal)", fontsize=11)
ax.set_title("Connectivity Tax  (N=6, 1 Trotter step)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("../results/plot2_tax_ratios.png", dpi=150)
plt.show()

---
## Plot 3 — Heat Maps: α vs Trotter Steps

Two panels showing 2Q depth across the full (α, steps) grid.
Reveals how the connectivity tax compounds with circuit depth.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, variant, title in [(ax1, "unitary", "Heavy-hex Unitary (SWAP)"),
                            (ax2, "dynamic", "Heavy-hex Dynamic (hybrid)")]:
    data = df[(df["variant"] == variant) & np.isclose(df["theta"], THETA_FIXED)]
    pivot = data.pivot_table(index="alpha", columns="n_steps",
                              values="two_q_depth", aggfunc="mean")
    alphas = pivot.index.values
    steps = pivot.columns.values
    Z = pivot.values

    im = ax.imshow(Z, aspect="auto", origin="lower",
                    extent=[steps.min()-0.5, steps.max()+0.5,
                            alphas.min()-0.15, alphas.max()+0.15],
                    cmap="YlOrRd", interpolation="nearest")
    ax.set_xlabel("Trotter Steps", fontsize=11)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax, label="Two-Qubit Depth")

    # Annotate a few values
    for ai, a in enumerate(alphas):
        for si, s in enumerate(steps):
            if s in [1, 4, 8] and a in [0.5, 2.0, 4.0]:
                val = Z[ai, si]
                if not np.isnan(val):
                    ax.text(s, a, f"{int(val)}", ha="center", va="center",
                            fontsize=7, color="black", fontweight="bold")

ax1.set_ylabel(r"$\alpha$", fontsize=12)
fig.suptitle("Two-Qubit Depth  (N=6)", fontsize=14)
fig.tight_layout()
fig.savefig("../results/plot3_heatmaps.png", dpi=150)
plt.show()

---
## Plot 4 — Phase Diagram (Feasibility Regions)

We define a **feasibility threshold** on 2Q depth.  On current IBM
hardware, a rough rule is that circuits with 2Q depth much beyond
~50–80 decohere badly (T2 ≈ 200 µs, CX time ≈ 0.3–0.6 µs, plus
idle/measurement overhead).  We use **2Q depth ≤ 50** as the threshold.

Three regions:
- **Green**: both unitary and dynamic within budget
- **Yellow**: only one variant within budget (dynamic saves depth for
  NN-dominated interactions)
- **Red**: both exceed budget — all-to-all connectivity needed

In [ ]:
THRESHOLD = 50  # 2Q depth budget

alphas = np.sort(df["alpha"].unique())
steps = np.sort(df["n_steps"].unique())

phase = np.full((len(alphas), len(steps)), np.nan)

for ai, a in enumerate(alphas):
    for si, s in enumerate(steps):
        u = df[(df["variant"]=="unitary") & (df["alpha"]==a) &
               (df["n_steps"]==s) & np.isclose(df["theta"], THETA_FIXED)]
        d = df[(df["variant"]=="dynamic") & (df["alpha"]==a) &
               (df["n_steps"]==s) & np.isclose(df["theta"], THETA_FIXED)]
        i = df[(df["variant"]=="ideal") & (df["alpha"]==a) &
               (df["n_steps"]==s) & np.isclose(df["theta"], THETA_FIXED)]

        uv = u["two_q_depth"].values[0] if len(u) > 0 else np.nan
        dv = d["two_q_depth"].values[0] if len(d) > 0 else np.nan
        iv = i["two_q_depth"].values[0] if len(i) > 0 else np.nan

        if np.isnan(uv):
            continue
        elif uv <= THRESHOLD and dv <= THRESHOLD:
            phase[ai, si] = 0  # green: both feasible
        elif uv > THRESHOLD and dv <= THRESHOLD:
            phase[ai, si] = 1  # yellow: only dynamic feasible
        elif uv <= THRESHOLD and dv > THRESHOLD:
            phase[ai, si] = 0.5  # light green: unitary OK, dynamic not
        else:
            phase[ai, si] = 2  # red: neither feasible

cmap_phase = mcolors.ListedColormap(["#4CAF50", "#81C784", "#FFC107", "#F44336"])
bounds = [-0.25, 0.25, 0.75, 1.5, 2.5]
norm = mcolors.BoundaryNorm(bounds, cmap_phase.N)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(phase, aspect="auto", origin="lower",
               extent=[steps.min()-0.5, steps.max()+0.5,
                       alphas.min()-0.15, alphas.max()+0.15],
               cmap=cmap_phase, norm=norm, interpolation="nearest")

# Draw threshold contour for unitary
u_pivot = df[(df["variant"]=="unitary") & np.isclose(df["theta"], THETA_FIXED)].pivot_table(
    index="alpha", columns="n_steps", values="two_q_depth", aggfunc="mean",
)
ax.contour(u_pivot.columns.values, u_pivot.index.values, u_pivot.values,
           levels=[THRESHOLD], colors=["white"], linewidths=2, linestyles=["--"])

legend_elements = [
    Patch(facecolor="#4CAF50", label="Both feasible"),
    Patch(facecolor="#FFC107", label="Only dynamic feasible"),
    Patch(facecolor="#F44336", label="Exceeds budget (need all-to-all)"),
    Patch(facecolor="none", edgecolor="white", linestyle="--", linewidth=2,
          label=f"Unitary 2Q depth = {THRESHOLD}"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9)
ax.set_xlabel("Trotter Steps", fontsize=12)
ax.set_ylabel(r"$\alpha$", fontsize=12)
ax.set_title(f"Feasibility Phase Diagram  (N=6, threshold={THRESHOLD})", fontsize=13)
fig.tight_layout()
fig.savefig("../results/plot4_phase_diagram.png", dpi=150)
plt.show()

---
## Plot 5 — Dynamic vs Unitary Overhead Ratio

Ratio d/u shows whether the dynamic approach saves depth compared
to pure SWAP routing.  d/u < 1 means dynamic wins; d/u > 1 means
SWAP routing is cheaper.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for n_steps_show in [1, 2, 4, 8]:
    sub = df[np.isclose(df["theta"], THETA_FIXED) & (df["n_steps"] == n_steps_show)]
    u = sub[sub["variant"]=="unitary"].set_index("alpha")["two_q_depth"]
    d = sub[sub["variant"]=="dynamic"].set_index("alpha")["two_q_depth"]
    ratio = d / u
    ratio = ratio.dropna().sort_index()
    ax.plot(ratio.index, ratio.values, "o-", label=f"steps={n_steps_show}", markersize=6)

ax.axhline(1.0, color="gray", ls=":", alpha=0.5, label="break-even")
ax.set_xlabel(r"$\alpha$", fontsize=12)
ax.set_ylabel("Dynamic / Unitary  (2Q depth ratio)", fontsize=11)
ax.set_title("Dynamic circuit overhead relative to SWAP routing  (N=6)", fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("../results/plot5_dynamic_vs_unitary.png", dpi=150)
plt.show()

---
## Summary Table

In [ ]:
summary = df[df["n_steps"] == 1].pivot_table(
    index="alpha",
    columns="variant",
    values=["two_q_depth", "two_q_count", "swap_count"],
    aggfunc="mean",
).round(0).astype(int)

print("=== Metrics at 1 Trotter step (N=6) ===\n")
print(summary.to_string())

---
## Key Findings

1. **The connectivity tax is real and measurable.**  For α ≤ 1.0 (strong
   long-range coupling), SWAP routing on heavy-hex inflates the 2Q depth
   compared to all-to-all connectivity.

2. **The tax grows with Trotter depth.**  As the number of Trotter steps
   increases, the overhead compounds — pushing circuits beyond typical
   coherence budgets at lower α values.

3. **Dynamic circuits eliminate SWAP overhead for NN interactions.**  By
   building the ancilla-assisted ZZ blocks directly on the heavy-hex
   bridge qubits, NN pairs incur zero routing cost.  At large α (NN-only
   regime), the dynamic circuit has no SWAPs at all.

4. **For long-range interactions, both approaches share the same routing
   cost.**  Non-NN pairs use bare Rzz gates that the transpiler routes
   with SWAPs — identical to the unitary variant.  The dynamic advantage
   comes purely from the NN portion.

5. **Topology matching determines dynamic circuit effectiveness.**  The
   advantage grows as more interactions are nearest-neighbour on the
   hardware.  This explains why the IBM tutorial's hexagonal model works
   so well on heavy-hex: all interactions are NN on the bridge topology.

6. **The feasibility boundary** (2Q depth ≤ threshold) shifts depending
   on which variant is used.  The dynamic approach can extend feasibility
   to deeper circuits or smaller α values when NN interactions dominate.